## Imports

In [126]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

In [127]:
from funcoes_escoragem import *

## Diretório

In [128]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [129]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [130]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) <= CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,765626977,4129365,2026-07-01,2026-07-01,1,NI,16988.000000000,BLEND3_3,4129365,"{""pessoas"":[{""nome"":""DANIELE DE FATIMA SEUGLIN...",...,32,1778785248777,"{""valor_aluguel"":1389.6,""matchmaking_on"":false...",2026-07-01 18:00:33+00:00,1782928828826724648,00765626977,B,2026-07-01 18:02:00.287000+00:00,2026-07-01 11:50:42+00:00,APROVAR
1,45270194813,4164901,2026-06-16,2026-06-16,1,B,5137.500000000,BLEND3_3,4164901,"{""pessoas"":[{""nome"":""ANTONIO BEZERRA DE LIMA J...",...,33,1778785248777,"{""valor_aluguel"":""2600"",""imobiliaria_id"":40682...",2026-06-17 08:00:35+00:00,1781683232212789699,45270194813,E,2026-06-17 08:04:49.943000+00:00,2026-06-16 16:17:32+00:00,APROVAR
2,69882380697,4196041,2026-06-21,2026-06-21,1,NI,32126.500000000,BLEND3_3,4196041,"{""pessoas"":[{""nome"":""MARY JUSSARA VIEIRA"",""doc...",...,32,1778785248777,"{""valor_aluguel"":""4400"",""imobiliaria_id"":52262...",2026-06-22 08:00:30+00:00,1782115227468405115,69882380697,B,2026-06-22 08:08:19.622000+00:00,2026-06-21 17:36:39+00:00,APROVAR
3,10845703943,4214582,2026-06-17,2026-06-17,1,C,1644.000000000,BLEND_REGRESSAO_2026,4214582,"{""pessoas"":[{""nome"":""ANA CLAUDIA BRITO ROCHA"",...",...,36,1778785248777,"{""valor_aluguel"":""1000"",""imobiliaria_id"":923,""...",2026-06-17 18:00:32+00:00,1781719230974853268,10845703943,D,2026-06-17 18:02:12.537000+00:00,2026-06-17 14:26:21+00:00,APROVAR
4,49552130816,4215975,2026-06-15,2026-06-16,1,B,1301.500000000,BLEND3_3,4215975,"{""pessoas"":[{""nome"":""GRAZIELE CASTILHO TEIXEIR...",...,34,1778785248777,"{""valor_aluguel"":""680"",""imobiliaria_id"":1179,""...",2026-06-16 18:00:37+00:00,1781632830647415232,49552130816,C,2026-06-16 18:03:52.170000+00:00,2026-06-16 08:46:52+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110463,4583242131,4354574,2026-07-09,2026-07-09,1,NI,3699.000000000,BLEND3_3,4354574,"{""pessoas"":[{""nome"":""JENNIFER SILVY RODRIGUES ...",...,34,1778785248777,"{""valor_aluguel"":1000,""matchmaking_on"":false,""...",2026-07-10 08:00:44+00:00,1783670442199960490,04583242131,C,2026-07-10 08:11:20.969000+00:00,2026-07-09 17:05:42+00:00,APROVAR
110464,86662422053,4354592,2026-07-09,2026-07-09,1,C,1918.000000000,BLEND3_3,4354592,"{""pessoas"":[{""nome"":""ALISSON DO NASCIMENTO"",""d...",...,33,1778785248777,"{""valor_aluguel"":1200,""matchmaking_on"":false,""...",2026-07-10 08:00:44+00:00,1783670442224859217,86662422053,E,2026-07-10 08:11:20.977000+00:00,2026-07-09 17:07:51+00:00,REPROVAR
110465,1039169970,4354927,2026-07-09,2026-07-09,1,A,3219.500000000,BLEND3_3,4354927,"{""pessoas"":[{""nome"":""TAISE DUMINELLI AMBONI"",""...",...,31,1778785248777,"{""valor_aluguel"":1800,""matchmaking_on"":false,""...",2026-07-10 08:00:44+00:00,1783670442791887148,01039169970,A,2026-07-10 08:11:21.132000+00:00,2026-07-09 18:09:49+00:00,APROVAR
110466,555965252,4355006,2026-07-09,2026-07-09,1,NI,5754.000000000,BLEND_4,4355006,"{""pessoas"":[{""nome"":""PATRICK AUGUSTO OLIVEIRA ...",...,31,1778785248777,"{""principalDocument"":""00555965252"",""imobiliari...",2026-07-10 08:00:44+00:00,1783670442934033005,00555965252,A,2026-07-10 08:11:21.170000+00:00,2026-07-09 19:23:20+00:00,APROVAR


In [131]:
        # CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,

In [164]:
credpago_df[credpago_df["contract_id"] == 4331926]

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
13982,31722504153,4331926,2026-07-03,2026-07-06,2,NI,49320.000000000,HVA3,4331926,"{""pessoas"":[{""nome"":""RAQUEL DA VEIGA ARAUJO DE...",...,33,1778785248777,"{""valor_aluguel"":5700,""matchmaking_on"":false,""...",2026-07-06 18:00:45+00:00,1783360841363090865,31722504153,E,2026-07-06 18:02:17.435000+00:00,2026-07-06 11:55:35+00:00,REPROVAR


In [163]:
credpago_clean[credpago_clean["contract_id"] == 4331926][["contract_id", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,contract_id,message_blend3Variables_cityPc4HistoryOver100Contracts
13982,4331926,NaN


In [134]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [135]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                94378
BLEND_REGRESSAO_2026     6897
HVA3                     4425
BLEND_4                  2420
BVS_CUSTOM               1522
HVA4                      709
BVS_CUSTOM_V2              94
                           15
HFT1                        8
Name: count, dtype: int64

## Multiproponente

In [136]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    106127
2      3996
3       316
4        26
8         1
5         1
6         1
Name: count, dtype: Int64

In [137]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.960704
2    0.036173
3    0.002861
4    0.000235
8    0.000009
5    0.000009
6    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON - Retornado

In [138]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [139]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

# json_normalize resets the index; preserve credpago_df labels for the join below
contrato_df = pd.json_normalize(payload_parsed.tolist(), sep="_")
contrato_df.index = payload_parsed.index
contrato_df = contrato_df.add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.

pessoa_records = payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0])
pessoa_df = pd.json_normalize(pessoa_records.tolist(), sep="_")
pessoa_df.index = payload_parsed.index
pessoa_df = pessoa_df.add_prefix("pessoa_")

In [140]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [141]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_valid = request_parsed.dropna()
request_df = pd.json_normalize(request_valid.tolist(), sep="_")
request_df.index = request_valid.index
request_df = request_df.add_prefix("request_")

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [142]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [143]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [144]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts
864,0.000000,0.189091
865,NaN,0.135476
890,NaN,0.076716
1961,0.000000,0.000000
1962,NaN,0.058575
...,...,...
109113,0.159091,0.096491
109131,0.000000,0.060096
109132,0.000000,0.080854
109135,NaN,NaN


## Escoragem Blend4

In [145]:
credpago_clean_rep = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [146]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [147]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [148]:
df_predict = (credpago_clean_rep.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] < 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_hadOverdueInvoiceInPreviousContracts,request_blend3Variables_cityPc4HistoryOver100Contracts,request_blend3Variables_realEstatePc4HistoryOver100Contracts,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,765626977,4129365,2026-07-01,2026-07-01,1,NI,BLEND3_3,6028763,32,00765626977,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,16988.0
1,45270194813,4164901,2026-06-16,2026-06-16,1,B,BLEND3_3,5957014,33,45270194813,...,NaN,NaN,NaN,2.0,5241.0,"CRED CARTAO,FINANCIAMENT",BLEND4,NaN,NaN,5137.5
2,69882380697,4196041,2026-06-21,2026-06-21,1,NI,BLEND3_3,5979250,32,69882380697,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,32126.5
3,10845703943,4214582,2026-06-17,2026-06-17,1,C,BLEND_REGRESSAO_2026,5962405,36,10845703943,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1644.0
4,49552130816,4215975,2026-06-15,2026-06-16,1,B,BLEND3_3,5951692,34,49552130816,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1301.5


In [149]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    110224
E_BVS        244
dtype: int64

In [150]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [151]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [152]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [153]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [154]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada.copy()
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,765626977,4129365,2026-07-01,2026-07-01,1,NI,BLEND3_3,6028763,32,00765626977,...,0.000000,0.80,0.0,-0.769018,0.000000,-0.615123,0.471046,529.0,A,A
1,45270194813,4164901,2026-06-16,2026-06-16,1,B,BLEND3_3,5957014,33,45270194813,...,0.000000,-0.40,3.0,0.168181,0.088511,-0.436110,0.545561,454.0,B,E
2,69882380697,4196041,2026-06-21,2026-06-21,1,NI,BLEND3_3,5979250,32,69882380697,...,0.000000,1.15,0.0,-0.647176,0.000000,0.000000,0.474788,525.0,A,A
3,10845703943,4214582,2026-06-17,2026-06-17,1,C,BLEND_REGRESSAO_2026,5962405,36,10845703943,...,0.000000,-0.30,0.0,0.393907,0.000000,0.000000,0.500147,500.0,A,E
4,49552130816,4215975,2026-06-15,2026-06-16,1,B,BLEND3_3,5951692,34,49552130816,...,0.000000,-0.80,0.0,0.204388,0.000000,-1.280311,0.471654,528.0,A,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110463,4583242131,4354574,2026-07-09,2026-07-09,1,NI,BLEND3_3,6074215,34,04583242131,...,0.000000,-0.05,0.0,-0.352543,0.000000,0.000000,0.482585,517.0,A,D
110464,86662422053,4354592,2026-07-09,2026-07-09,1,C,BLEND3_3,6074235,33,86662422053,...,0.000000,-0.25,14.0,0.432296,-0.057882,-0.488031,0.639243,361.0,D,E
110465,1039169970,4354927,2026-07-09,2026-07-09,1,A,BLEND3_3,6074647,31,01039169970,...,0.000000,0.05,0.0,0.285275,-0.028211,0.544700,0.570941,429.0,C,A
110466,555965252,4355006,2026-07-09,2026-07-09,1,NI,BLEND_4,6074749,31,00555965252,...,0.574344,-0.05,0.0,-0.143537,0.000000,2.460269,0.424355,576.0,A,A


In [155]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           15
BLEND3_3                94378
BLEND_4                  2420
BLEND_REGRESSAO_2026     6897
BVS_CUSTOM               1522
BVS_CUSTOM_V2              94
HFT1                        8
HVA3                     4425
HVA4                      709
dtype: int64

In [156]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [157]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    110224
E_BVS        244
dtype: int64

In [158]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           15
BLEND3_3                94378
BLEND_4                  2420
BLEND_REGRESSAO_2026     6897
BVS_CUSTOM               1522
BVS_CUSTOM_V2              94
HFT1                        8
HVA3                     4425
HVA4                      709
dtype: int64

In [159]:
cp_final_df.groupby("model", dropna=False).size()

model
                           15
BLEND3_3                94378
BLEND_4                  2420
BLEND_REGRESSAO_2026     6897
BVS_CUSTOM               1522
BVS_CUSTOM_V2              94
HFT1                        8
HVA3                     4425
HVA4                      709
dtype: int64

In [160]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    110224
E_BVS        244
dtype: int64

In [161]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)

## Investigação Sexta-Feira (03/07)

In [162]:
cp_final_df[
    (pd.to_datetime(cp_final_df["requested_at"]).dt.normalize() == "2026-07-03")
    & (cp_final_df["message_decisao"] == "BLEND_4")
    & (cp_final_df["pred_blend4_1_to_score"] != cp_final_df["message_blendRegressaoPredict"])
][id_vars + vars_blend4 + ["pred_blend4_1_to_score", "message_blendRegressaoPredict"]]

,contract_id,requested_at,CPF_CNPJ,message_decisao,message_blendRegressaoPredict,score_proposto__bvs,SERASA_CHSV5,age,property_type,qtde_restricoes__consulta_realizada,rental_value,income,agency_pc4_mais_100_contratos__pc_categorias,city_pc4_mais_100_contratos__pc_categorias,flag_tem__contratos_anteriores,flag_teve_boleto_atrasado__contratos_anteriores,pred_blend4_1_to_score,message_blendRegressaoPredict
864,4329715,2026-07-03,41110509804,BLEND_4,162.0,381.0,83.0,19.0,0,1,1700.0,1712.5,0.00000,0.189091,0.0,0.0,144.0,162.0
1961,4329656,2026-07-03,5240159106,BLEND_4,667.0,574.0,782.0,23.0,1,0,1700.0,1781.0,0.00000,0.000000,0.0,0.0,704.0,667.0
1978,4331939,2026-07-03,49941717850,BLEND_4,440.0,529.0,420.0,23.0,1,0,1600.0,1507.0,0.00000,0.120879,0.0,0.0,407.0,440.0
5225,4331789,2026-07-03,4240289300,BLEND_4,454.0,471.0,430.0,44.0,1,0,1050.0,2055.0,0.00000,0.094955,0.0,0.0,421.0,454.0
6321,4329890,2026-07-03,46224044864,BLEND_4,587.0,578.0,571.0,26.0,1,0,1300.0,1438.5,0.00000,0.000000,0.0,0.0,628.0,587.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106825,4332279,2026-07-03,945694288,BLEND_4,359.0,528.0,301.0,32.0,1,2,2500.0,2055.0,0.00000,0.130864,0.0,0.0,328.0,359.0
106826,4332330,2026-07-03,11970133694,BLEND_4,NaN,182.0,468.0,34.0,1,0,850.0,0.0,0.00000,0.000000,0.0,0.0,322.0,NaN
109131,4331830,2026-07-03,1436871158,BLEND_4,671.0,576.0,711.0,39.0,0,0,3000.0,14727.5,0.00000,0.060096,0.0,0.0,641.0,671.0
109132,4331866,2026-07-03,10255326947,BLEND_4,807.0,689.0,960.0,30.0,1,0,5200.0,9453.0,0.00000,0.080854,0.0,0.0,786.0,807.0


## Teste